# Step 1: Research & Data Source Discovery

## 1.1 Environment Setup and Dependencies

We fetch the data from OpenStreetMap. We use the original OSM ID (osmid) as our primary identifier and calculate the exact center point (latitude and longitude) for each location.

* **Primary Source: OpenStreetMap (OSM)**: Used to extract the spatial location of employment agencies.


## 1.2 Data and Boundary Configuration

The project focuses exclusively on data within the **Berlin, Germany** boundary.

* **Spatial Integrity Plan**: Data will be joined to the **Local Reference System (LOR) boundaries** to derive the mandatory `district_id` and `neighborhood_id` for final database compliance.

In [14]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import os 
import json
!pip install geopy
%pip install sqlalchemy psycopg2-binary
import psycopg2
from sqlalchemy import create_engine, text
import warnings

# --- 1.1 CONFIGURATION ---
# Using the specific paths and tags from your previous workflow
PLACE_NAME = "Berlin, Germany"
OSM_TAGS = {"office": "employment_agency"}

# Update LOR_PATH to match the exact filename of the GeoJSON you uploaded
#LOR_PATH = "lor_ortsteile (1).geojson" 
#OUTPUT_PATH = "output/jobcenters_berlin.csv"

print("Libraries loaded.")
print(f"Configuration set for {PLACE_NAME} with OSM tags: {OSM_TAGS}")

# --- 1.2 LIVE DATA EXTRACTION (OSM) ---
print("\nFetching live data from OpenStreetMap (Overpass API)...")
try:
    # Fetch data and ensure the coordinate system is standard WGS84 (EPSG:4326)
    jobcenter_data_raw = ox.features_from_place(PLACE_NAME, OSM_TAGS)
    jobcenter_data_raw = gpd.GeoDataFrame(
        jobcenter_data_raw,
        geometry="geometry",
        crs="EPSG:4326"
    )
    print(f"Success! Retrieved {len(jobcenter_data_raw)} features.")
except Exception as e:
    raise RuntimeError(f"OSM extraction failed: {e}")

# --- 1.3 MANDATORY DATA CLEANING ---
# We explicitly check and report on null values in mandatory columns
print("\n--- Diagnostic Check: Nulls in Critical Columns ---")
null_counts = jobcenter_data_raw[['name', 'geometry']].isnull().sum()
print("Missing values in critical columns:")
print(null_counts)

# Drop rows missing 'name' or 'geometry' to enforce database NOT NULL compliance
initial_count = len(jobcenter_data_raw)
jobcenter_enriched = jobcenter_data_raw.dropna(subset=["name", "geometry"]).copy()

dropped_count = initial_count - len(jobcenter_enriched)
print(f"Mandatory Drop: Removed {dropped_count} rows due to missing name/geometry.")

# --- 1.4 COORDINATE PREPARATION ---
# Extract centroids to handle both 'Point' and 'Polygon' features safely
jobcenter_enriched['latitude'] = jobcenter_enriched.geometry.centroid.y
jobcenter_enriched['longitude'] = jobcenter_enriched.geometry.centroid.x

print("\n--- Step 1 Complete ---")
print(jobcenter_enriched[['name', 'latitude', 'longitude']].head())

Note: you may need to restart the kernel to use updated packages.
Libraries loaded.
Configuration set for Berlin, Germany with OSM tags: {'office': 'employment_agency'}

Fetching live data from OpenStreetMap (Overpass API)...
Success! Retrieved 65 features.

--- Diagnostic Check: Nulls in Critical Columns ---
Missing values in critical columns:
name        2
geometry    0
dtype: int64
Mandatory Drop: Removed 2 rows due to missing name/geometry.

--- Step 1 Complete ---
                                               name   latitude  longitude
element id                                                               
node    275368512   Jobcenter Mitte am Leopoldplatz  52.546772  13.356516
        1211913324           Arbeitsagentur Spandau  52.533775  13.186554
        1340158173        Jobcenter Berlin Neukölln  52.478975  13.427887
        1450906609               Agentur für Arbeit  52.578452  13.308718
        2277566662               Agentur für Arbeit  52.456592  13.411478


/var/folders/w5/hrcdt9v17ws_d71g1h0pcwjc0000gn/T/ipykernel_1959/234890649.py:54: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  jobcenter_enriched['latitude'] = jobcenter_enriched.geometry.centroid.y
/var/folders/w5/hrcdt9v17ws_d71g1h0pcwjc0000gn/T/ipykernel_1959/234890649.py:55: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  jobcenter_enriched['longitude'] = jobcenter_enriched.geometry.centroid.x


## Step 2: Cleanup & Removing Redundancy
Explanation: Here we drop city and country because they are redundant for a Berlin project. We also remove contact:website and operator:type to keep the schema lean.
Why drop contact "website"

Maintenance: External URLs like websites change frequently. If you include them in the primary table now, the data becomes "stale" very quickly.

Scope: The current goal is to map the job centers to the Berlin District LOR system. Extra information like websites or phone numbers can be added in a later "enrichment" task once the primary table structure is approved.

Additionally: The operator:type column is a classification tag in OpenStreetMap. It tells the database who runs the facility. In the context of Berlin Job Centers, this usually indicates public. 
The center is a government-run entity (e.g., the Bundesagentur für Arbeit or local municipal government). Most Job Centers fall into this category.

In [15]:
from geopy.geocoders import Nominatim
from time import sleep

# 1. INITIAL CLEANUP: Rename and prepare coordinates
# Prefer the cleaned/enriched frame from earlier; fall back to the raw OSM frame
if 'jobcenter_enriched' in globals():
    jobcenter_clean = jobcenter_enriched.copy()
elif 'jobcenter_data_raw' in globals():
    jobcenter_clean = jobcenter_data_raw.copy()
else:
    raise NameError("Expected 'jobcenter_enriched' or 'jobcenter_data_raw' to be defined.")
jobcenter_clean = jobcenter_clean.rename(columns={'name': 'center_name'})

# Calculate centroids to ensure we have lat/lon for both Points and Polygons
centroids = jobcenter_clean.geometry.centroid
jobcenter_clean['latitude'] = centroids.y
jobcenter_clean['longitude'] = centroids.x

# 2. BUILD ADDRESS FROM COLUMNS (The Boss's First Priority)
# Using fillna('') to avoid "NaN" appearing in the text strings
jobcenter_clean['address'] = (
    jobcenter_clean['addr:street'].fillna('') + ' ' + 
    jobcenter_clean['addr:housenumber'].fillna('')
).str.strip()

# Add house name in brackets if it exists (e.g., "Jobcenter Mitte")
mask_housename = jobcenter_clean['addr:housename'].notna()
jobcenter_clean.loc[mask_housename, 'address'] = (
    jobcenter_clean['address'] + ' (' + jobcenter_clean['addr:housename'] + ')'
).str.strip()

# Map the postal code from OSM
jobcenter_clean['postal_code'] = jobcenter_clean['addr:postcode']

# 3. NOMINATIM FALLBACK (The Boss's Second Priority)
geolocator = Nominatim(user_agent="berlin_jobcenter_locator")

def get_nominatim_data(lat, lon):
    """Retrieves both address and postcode from Nominatim"""
    try:
        location = geolocator.reverse((lat, lon), exactly_one=True, language='de')
        sleep(1) # Crucial: Respect Nominatim's 1-second rate limit
        if location:
            address_text = location.address
            postcode = location.raw.get('address', {}).get('postcode')
            return address_text, postcode
        return None, None
    except:
        return None, None

# Find rows where address is still empty OR postal_code is NaN
mask_missing = (jobcenter_clean['address'] == "") | (jobcenter_clean['postal_code'].isna())

if mask_missing.any():
    print(f"🔍 Found {mask_missing.sum()} rows needing Nominatim enrichment. Starting fallback...")
    
    # We apply the function to fill both columns at once
    results = jobcenter_clean[mask_missing].apply(
        lambda row: get_nominatim_data(row['latitude'], row['longitude']), axis=1
    )
    
    # Extract the results back into the dataframe
    jobcenter_clean.loc[mask_missing, 'address'] = [r[0] for r in results]
    jobcenter_clean.loc[mask_missing, 'postal_code'] = [r[1] for r in results]
else:
    print(" All addresses and postal codes were successfully built from existing data!")

print("\n--- Verification of Enriched Data ---")
print(jobcenter_clean[['center_name', 'address', 'postal_code']].head())

🔍 Found 14 rows needing Nominatim enrichment. Starting fallback...


/var/folders/w5/hrcdt9v17ws_d71g1h0pcwjc0000gn/T/ipykernel_1959/2076044463.py:15: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = jobcenter_clean.geometry.centroid



--- Verification of Enriched Data ---
                                        center_name  \
element id                                            
node    275368512   Jobcenter Mitte am Leopoldplatz   
        1211913324           Arbeitsagentur Spandau   
        1340158173        Jobcenter Berlin Neukölln   
        1450906609               Agentur für Arbeit   
        2277566662               Agentur für Arbeit   

                                                              address  \
element id                                                              
node    275368512   Müllerstraße 147 (Jobcenter Mitte am Leopoldpl...   
        1211913324                           Brunsbütteler Damm 75-77   
        1340158173                                  Mainzer Straße 27   
        1450906609                                   Innungsstraße 40   
        2277566662  Agentur für Arbeit, 43-44, Gottlieb-Dunkel-Str...   

                   postal_code  
element id                    

## Step 3: Spatial Mapping (District Join)

Explanation: Load the official Berlin district file and  use a Spatial Join to see which district polygon each job center point "falls into." This gives us the neighborhood and district names automatically.

In [16]:
LOR_PATH = "lor_ortsteile.geojson"
lor_gdf = gpd.read_file(LOR_PATH).to_crs(epsg=4326)

In [17]:
import os
print("LOR file exists:", os.path.exists(LOR_PATH))

LOR file exists: True


In [18]:
# 1. Safe Renaming: Only rename if the old columns still exist
if "BEZIRK" in lor_gdf.columns:
    lor_gdf = lor_gdf.rename(columns={
        "BEZIRK": "district",
        "OTEIL": "neighborhood",
        "spatial_name": "neighborhood_id"
    })

# 2. Safety check: Remove 'index_right' to prevent the SJOIN ValueError
if 'index_right' in jobcenter_clean.columns:
    jobcenter_clean = jobcenter_clean.drop(columns=['index_right'])

# 3. Spatial Join
jobcenter_mapped = gpd.sjoin(
    jobcenter_clean.reset_index(drop=True), 
    lor_gdf[['district', 'neighborhood', 'neighborhood_id', 'geometry']], 
    how='left', 
    predicate='within'
)

print("Join completed successfully (even on a re-run!)")

Join completed successfully (even on a re-run!)


## 4: Stable ID Generation and District Mapping
Deterministic Stable ID: A persistent, numeric-only ID is generated using hashlib.sha256. By hashing the geographic centroid, we ensure IDs are unique and immutable, avoiding previous AttributeError issues with different geometry types.

Official District Mapping: To comply with the final data pool schema, we map administrative district names to their official 8-digit numeric IDs (e.g., Mitte = 11001001). This ensures the data is ready for SQL relational joins.hment:** The `enrich_data_from_wikidata` function is applied to fill the `operator_name` and `contact_website` columns.

In [19]:
import hashlib

# --- 4.1 DEFINITIONS ---
def generate_stable_id(name, lat, lon):
    """Generates a unique 10-digit ID based on name and coordinates."""
    input_data = f"{name}_{lat}_{lon}".encode('utf-8')
    hash_hex = hashlib.sha256(input_data).hexdigest()
    return int(hash_hex, 16) % (10**10)

district_mapping = {
    'Mitte': '11001001', 'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003', 'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005', 'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007', 'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009', 'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011', 'Reinickendorf': '11012012'
}

# --- 4.2 EXECUTION (The Calls) ---
# 1. Coordinate Prep (Ensuring columns exist)
jobcenter_mapped['latitude'] = jobcenter_mapped.geometry.centroid.y
jobcenter_mapped['longitude'] = jobcenter_mapped.geometry.centroid.x

# 2. Call the Stable ID function
print("Generating stable IDs...")
jobcenter_mapped['id'] = jobcenter_mapped.apply(
    lambda row: generate_stable_id(row['center_name'], row['latitude'], row['longitude']), 
    axis=1
)

# 3. Call the District mapping
print("Mapping districts...")
jobcenter_mapped['district_id'] = jobcenter_mapped['district'].map(district_mapping)

print("Step 4 complete. Data is enriched and identified.")
print("Mapping district names to official IDs...")
jobcenter_mapped['district_id'] = jobcenter_mapped['district'].map(district_mapping).astype(str)

# --- Verification ---
print("\n--- Step 4 Verification ---")
print(jobcenter_mapped[['id', 'center_name', 'district', 'district_id']].head())

Generating stable IDs...
Mapping districts...
Step 4 complete. Data is enriched and identified.
Mapping district names to official IDs...

--- Step 4 Verification ---
           id                      center_name              district  \
0  6660665090  Jobcenter Mitte am Leopoldplatz                 Mitte   
1  1092468394           Arbeitsagentur Spandau               Spandau   
2   730832232        Jobcenter Berlin Neukölln              Neukölln   
3   246338546               Agentur für Arbeit         Reinickendorf   
4  6239357044               Agentur für Arbeit  Tempelhof-Schöneberg   

  district_id  
0    11001001  
1    11005005  
2    11008008  
3    11012012  
4    11007007  


/var/folders/w5/hrcdt9v17ws_d71g1h0pcwjc0000gn/T/ipykernel_1959/347821576.py:21: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  jobcenter_mapped['latitude'] = jobcenter_mapped.geometry.centroid.y
/var/folders/w5/hrcdt9v17ws_d71g1h0pcwjc0000gn/T/ipykernel_1959/347821576.py:22: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  jobcenter_mapped['longitude'] = jobcenter_mapped.geometry.centroid.x


## 5: Data Standardization and Final Export
Schema Compliance: The final dataset is filtered to include only the mandatory 8 columns required for the database pool: id, district_id, center_name, latitude, longitude, neighborhood, district, and neighborhood_id.

WKT & Coordinate Prep: Coordinates are extracted from the geometric centroids and formatted as numeric floats, ensuring compatibility with standard SQL spatial types.

Stable ID Integration: The deterministic IDs generated in Step 4 are finalized as the primary keys for this dataset.

Data Source Attribution: A data_source tag (OSM_LOR) is appended to ensure traceability for future audits.

In [20]:
import os

# --- 5.1 SPATIAL DATA PREP ---
# Remove rows without shapes to prevent database errors
df_ready = jobcenter_mapped.dropna(subset=['geometry']).copy()

# 2. Translate Geometry to WKT (Only if it's not already text)
# This allows the Python 'to_sql' command to send the data successfully
def serialize_geometry(x):
    if hasattr(x, 'wkt'):
        return x.wkt
    return x

df_ready['geometry'] = df_ready['geometry'].apply(serialize_geometry)

# --- 5.2 PRODUCTION SCHEMA SELECTION ---
# We use the exact names required by the database
final_columns = [
    'id', 'district_id', 'center_name', 'address', 'postal_code', 
    'latitude', 'longitude', 'geometry', 'neighborhood', 
    'district', 'neighborhood_id'
]

# Create the final dataframe and add the data source tag
df_final = df_ready[[c for c in final_columns if c in df_ready.columns]].copy()
df_final['data_source'] = 'OSM_LOR'

# --- 5.3 EXPORT ---
os.makedirs("output", exist_ok=True)
output_path = "output/jobcenters_berlin_final.csv"
df_final.to_csv(output_path, index=False)

print(f"Success: {len(df_final)} records organized and saved to {output_path}")

Success: 63 records organized and saved to output/jobcenters_berlin_final.csv


/var/folders/w5/hrcdt9v17ws_d71g1h0pcwjc0000gn/T/ipykernel_1959/367716047.py:14: UserWarning: Geometry column does not contain geometry.
  df_ready['geometry'] = df_ready['geometry'].apply(serialize_geometry)


Configuration 

In [21]:
user_name=''
password=''

Configuration 

In [22]:
user_name = 'tigist_hayilemariyam'
password = 'tc3WUbE1DZ6SzYZ'
host = '127.0.0.1' 
port = '5433'
database = 'layereddb'
schema = 'berlin_source_data'
table_name = 'job_centers'

Database Engine Initialization

In [23]:
engine = create_engine(f'postgresql+psycopg2://{user_name}:{password}@{host}:{port}/{database}')

### 6: Schema Definition & Constraints
6.1 We define the table structure with the following safeguards:

6.2 Primary Key: Ensures every Job Center has a unique, non-duplicate ID.

6.3 Foreign Key (FK): Links the district_id to the official city districts table. This prevents "orphan records" and ensures every center is assigned to a valid Berlin district.

6.4 Data Integrity: Sets specific formats for coordinates (Double Precision) and administrative names (Text).

In [24]:
from sqlalchemy import text

# 1. The production blueprint table creation 
# We use 'district_id' for the reference as it's the standard for this database
create_table_query = """
DROP TABLE IF EXISTS berlin_source_data.job_centers CASCADE;

CREATE TABLE berlin_source_data.job_centers (
    id TEXT PRIMARY KEY,
    district_id TEXT NOT NULL,
    center_name TEXT,
    address TEXT,
    postal_code TEXT,
    latitude DOUBLE PRECISION,
    longitude DOUBLE PRECISION,
    geometry TEXT,
    neighborhood TEXT,
    district TEXT,
    neighborhood_id TEXT,
    data_source TEXT,
    CONSTRAINT fk_district FOREIGN KEY (district_id) 
        REFERENCES berlin_source_data.districts (district_id) -- Matching the LOR standard
);
"""

# 2. Execute Table Creation
with engine.connect() as conn:
    conn.execute(text(create_table_query))
    conn.commit()
    print(" SUCCESS: The table 'job_centers' has been created!")

#  3. Final Production Upload
#  I use 'append' because the table was freshly created in the previous step
df_final.to_sql(
    name='job_centers',
    con=engine,
    schema='berlin_source_data',
    if_exists='append', 
    index=False
)

print(f" Success: {len(df_final)} records successfully deployed to AWS Production.")


 SUCCESS: The table 'job_centers' has been created!
 Success: 63 records successfully deployed to AWS Production.


In [25]:
# This query asks for every single row
full_check_query = "SELECT * FROM berlin_source_data.job_centers ORDER BY district_id;"

with engine.connect() as conn:
    df_all = pd.read_sql(text(full_check_query), conn)

# This tells the notebook to show all 63 rows without hiding any
with pd.option_context('display.max_rows', None):
    display(df_all)

,id,district_id,center_name,address,postal_code,latitude,longitude,geometry,neighborhood,district,neighborhood_id,data_source
0,2782312819,11001001,Job-Point,Alt-Moabit 84,10555,52.525535,13.339522,POINT (13.3395222 52.5255348),Moabit,Mitte,0102,OSM_LOR
1,4971452348,11001001,Kraftfahrer-Agentur,Barfusstraße 15,13349,52.555671,13.346877,POINT (13.3468772 52.5556712),Wedding,Mitte,0105,OSM_LOR
2,1346016268,11001001,Zenjob,"Zenjob, Stromstraße, Alt-Moabit, Moabit, Mitte...",10555,52.527579,13.343648,POINT (13.3436479 52.5275788),Moabit,Mitte,0102,OSM_LOR
3,6660665090,11001001,Jobcenter Mitte am Leopoldplatz,Müllerstraße 147 (Jobcenter Mitte am Leopoldpl...,13353,52.546772,13.356516,POINT (13.3565162 52.5467722),Wedding,Mitte,0105,OSM_LOR
4,3137117730,11001001,BSM Personalmanagement,"BSM Personalmanagement, Lehrter Straße, Moabit...",10557,52.535615,13.357524,POINT (13.3575242 52.5356149),Moabit,Mitte,0102,OSM_LOR
5,537096787,11001001,Beta gGmbH,Maxstraße 20,13347,52.549239,13.364556,POINT (13.3645563 52.5492387),Wedding,Mitte,0105,OSM_LOR
6,3608024519,11001001,recrew,"recrew, 44, Georgenstraße, Dorotheenstadt, Mit...",10117,52.520063,13.393426,POINT (13.3934256 52.5200629),Mitte,Mitte,0101,OSM_LOR
7,5424095615,11001001,Jobcenter Berlin Mitte,Müllerstraße 16,13353,52.543833,13.365058,"POLYGON ((13.3653305 52.5435849, 13.3657019 52...",Wedding,Mitte,0105,OSM_LOR
8,7211656755,11001001,Agentur Schlag,Joseph-Haydn-Straße 1,10557,52.514923,13.337123,POINT (13.3371231 52.5149233),Hansaviertel,Mitte,0103,OSM_LOR
9,9462705127,11001001,Players Agentur Management,Sophienstraße 21,10178,52.525917,13.400791,POINT (13.4007908 52.525917),Mitte,Mitte,0101,OSM_LOR


In [26]:
query = """
SELECT district, COUNT(*) as total 
FROM berlin_source_data.job_centers 
GROUP BY district 
ORDER BY total DESC;
"""

with engine.connect() as conn:
    df_results = pd.read_sql(text(query), conn)

print(df_results)

                      district  total
0                        Mitte     14
1   Charlottenburg-Wilmersdorf     10
2     Friedrichshain-Kreuzberg      9
3                     Neukölln      7
4                       Pankow      4
5                      Spandau      4
6         Tempelhof-Schöneberg      4
7          Steglitz-Zehlendorf      4
8          Marzahn-Hellersdorf      3
9             Treptow-Köpenick      2
10                 Lichtenberg      1
11               Reinickendorf      1
